# FOSAE-Compatible Object Detection Dataset Generation with Unsloth VLMs

This notebook provides a clean, modern pipeline for generating labeled object detection datasets using Unsloth vision-language models (e.g., Qwen3-VL, Llama-3-V) and Florence-2. The output is tailored for FOSAE predicate learning.

**Pipeline Steps:**
1. Environment & Dependency Setup
2. Dataset Loading (COCO or custom images)
3. Unsloth VLM Setup (Qwen3-VL, etc.)
4. Batch Annotation Pipeline
5. Output Parsing to FOSAE Format
6. Metrics & Visualization
7. Export for FOSAE Training

**References:**
- [Unsloth Vision RL Guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide/vision-reinforcement-learning-vlm-rl)
- [Qwen3-VL Fine-tuning](https://unsloth.ai/docs/models/qwen3-how-to-run-and-fine-tune/qwen3-vl-how-to-run-and-fine-tune#fine-tuning-qwen3-vl)
- [FOSAE Predicate Learning](https://arxiv.org/abs/1902.08093)

In [1]:
import os, re
# Do this only in Colab notebooks! Otherwise use pip install unsloth
import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.26.2
!pip install fiftyone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.3.2 requires msgspec, which is not installed.
unsloth-zoo 2026.3.2 requires tyro, which is not installed.
unsloth-zoo 2026.3.2 requires t

## 2. Dataset Loading (COCO or Custom Images)

Load a subset of the COCO 2017 dataset (or your own images) using FiftyOne. For custom images, place them in a directory and update the path below.

In [2]:
import os
from unsloth import FastVisionModel
import matplotlib.pyplot as plt
import numpy as np
import fiftyone as fo
import fiftyone.zoo as foz
from PIL import Image
import glob
import json
import torch

SAMPLE_SIZE = 5000

try:
    dataset = foz.load_zoo_dataset(
        "coco-2017",
        split="train",
        max_samples=SAMPLE_SIZE,
        seed=42,
        shuffle=True
    )
    print(f"Loaded COCO dataset with {len(dataset)} samples.")
except Exception as e:
    print("Could not load COCO dataset. Please check your internet connection or dataset path.")
    print(e)
    dataset = None

# For custom images, use:
# dataset = fo.Dataset.from_dir(dataset_dir="./your_images", dataset_type=fo.types.ImageDirectory)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/coco-2017/train' if necessary


INFO:fiftyone.utils.coco:Downloading annotations to '/root/fiftyone/coco-2017/tmp-download/annotations_trainval2017.zip'


 100% |██████|    1.9Gb/1.9Gb [7.7s elapsed, 0s remaining, 268.4Mb/s]       


INFO:eta.core.utils: 100% |██████|    1.9Gb/1.9Gb [7.7s elapsed, 0s remaining, 268.4Mb/s]       


...

 100% |███████████████| 5000/5000 [31.7s elapsed, 0s remaining, 196.9 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 5000/5000 [31.7s elapsed, 0s remaining, 196.9 samples/s]      


Dataset 'coco-2017-train-5000' created


INFO:fiftyone.zoo.datasets:Dataset 'coco-2017-train-5000' created


Loaded COCO dataset with 5000 samples.


## 3. Unsloth VLM Setup (Qwen3-VL, Llama-3-V, etc.)

In [3]:
from unsloth import FastVisionModel
import torch
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    gpu_memory_utilization = 0.8, # Reduce if out of memory

)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

## 4. Batch Annotation Pipeline (Open-Vocabulary Detection)

In [ ]:
RAW_IMAGE_DIR = "./raw_images"
OUTPUT_JSON = "unsloth_vlm_annotations.json"

os.makedirs(RAW_IMAGE_DIR, exist_ok=True)
if dataset is not None:
    for sample in dataset:
        img_path = sample.filepath
        img_name = os.path.basename(img_path)
        dest_path = os.path.join(RAW_IMAGE_DIR, img_name)
        if not os.path.exists(dest_path):
            os.symlink(img_path, dest_path)

results = []
image_files = sorted(glob.glob(os.path.join(RAW_IMAGE_DIR, "*.jpg")))

for img_path in image_files:
    image = Image.open(img_path).convert("RGB")
    prompt = """Detect all objects and label them with their class names. You MUST give the most specific class name you can for each object. You MUST output the result strictly as a JSON array of dictionaries. Each dictionary must contain exactly two keys:- "class": A string representing the object's label.- "bbox_2d": An array of four coordinates [xmin, ymin, xmax, ymax].Do not include any conversational text, markdown formatting outside the JSON, or explanations."""    
    MIN_PIXELS = 256 * 28 * 28
    MAX_PIXELS = 1280 * 28 * 28 

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image", 
                    "image": img_path,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {"type": "text", "text": prompt}
            ]
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        output = model.generate(**inputs,max_new_tokens=256,temperature=0.7,top_p=0.8,repetition_penalty=1.0,use_cache=True)
    results.append({
        "image": os.path.basename(img_path),
        "output": tokenizer.decode(output[0]) if hasattr(output, '__getitem__') else str(output)
    })

with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved Unsloth VLM raw outputs to {OUTPUT_JSON}")

## 5. Output Parsing to FOSAE Format

In [9]:
import re
import json

def parse_unsloth_output(output):
    objects = []
    if isinstance(output, (list, tuple)):
        output = output[0]
    if isinstance(output, bytes):
        output = output.decode("utf-8")
    if not isinstance(output, str):
        return objects
        
    # Remove special tokens and code block markers
    output = re.sub(r"<\|im_start\|>.*?assistant", "", output, flags=re.DOTALL)
    output = output.replace("```json", "").replace("```", "").strip()
    
    # Extract all complete dictionary structures, ignoring the outer array.
    # The regex \{[^{}]*\} matches '{', followed by anything that isn't a brace, ending with '}'.
    # This completely solves the non-greedy bracket issue AND ignores truncated ends!
    dict_strings = re.findall(r'\{[^{}]*\}', output)
    
    if not dict_strings:
        print(f"No JSON objects found in output: {output[:200]}...")
        return objects

    for json_str in dict_strings:
        try:
            # Clean up potential trailing commas inside the dict just in case
            clean_str = re.sub(r',\s*\}', '}', json_str)
            obj = json.loads(clean_str)
            
            label = obj.get("label") or obj.get("class")
            bbox = obj.get("bbox_2d") or obj.get("bbox")
            
            if label is not None and bbox is not None:
                objects.append({"class": label, "bbox": bbox})
        except Exception as e:
            # If a single object is malformed, skip it and continue parsing the rest
            print(f"JSON parse error on object: {e}\nOutput: {json_str[:100]}...")
            
    return objects

# --- Main Execution ---
with open("unsloth_vlm_annotations.json", "r") as f:
    results = json.load(f)

fosae_data = []
for result in results:
    parsed = parse_unsloth_output(result["output"])
    fosae_data.append({
        "image": result["image"],
        "objects": parsed
    })

with open("fosae_labeled_dataset_unsloth.json", "w") as f:
    json.dump(fosae_data, f, indent=2)
print("Exported FOSAE-compatible labeled dataset (Unsloth) to fosae_labeled_dataset_unsloth.json")

## 6. Metrics & Visualization

In [16]:
from collections import Counter
import matplotlib.pyplot as plt

# 1. Calculate metrics
object_counts = [len(item["objects"]) for item in fosae_data]
label_counter = Counter()
for item in fosae_data:
    for obj in item["objects"]:
        label_counter[obj["class"]] += 1

print(f"Total images: {len(fosae_data)}")
print(f"Total objects: {sum(object_counts)}")
print("Top 10 Label distribution:")
for label, count in label_counter.most_common(10):
    print(f"  {label}: {count}")

# 2. Setup Figure for side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Object Label Distribution (Top 20, Sorted) ---
top_labels = label_counter.most_common(20)
if top_labels:
    labels, counts = zip(*top_labels)
    axes[0].bar(labels, counts, color='steelblue', edgecolor='black')
    axes[0].set_title("Top 20 Object Class Distribution", fontsize=14)
    axes[0].set_xlabel("Class Label", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    
    axes[0].set_xticks(range(len(labels)))
    axes[0].set_xticklabels(labels, rotation=45, ha='right')
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# --- Plot 2: Objects per Image Histogram ---
if object_counts:
    max_objs = max(object_counts)
    bins = range(0, max_objs + 2) 
    axes[1].hist(object_counts, bins=bins, align='left', color='mediumseagreen', edgecolor='black')
    axes[1].set_title("Distribution of Objects per Image", fontsize=14)
    axes[1].set_xlabel("Number of Objects Detected", fontsize=12)
    axes[1].set_ylabel("Number of Images", fontsize=12)
    
    axes[1].set_xticks(range(0, max_objs + 1, max(1, max_objs // 10))) 
    axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()

# Save the metrics chart
METRICS_SAVE_PATH = "dataset_metrics.png"
plt.savefig(METRICS_SAVE_PATH, bbox_inches='tight', dpi=300)
print(f"Saved metrics visualization to '{METRICS_SAVE_PATH}'")

plt.show()
plt.close()

In [17]:
import os

# Create a directory to save the annotated images
ANNOTATED_DIR = "./annotated_images"
os.makedirs(ANNOTATED_DIR, exist_ok=True)

# Visualization: Bounding Box Overlay
def plot_image_with_boxes(image_path, objects, save_dir=None):
    image = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(6,6))
    plt.imshow(image)
    ax = plt.gca()
    for obj in objects:
        bbox = obj["bbox"]
        label = obj["class"]
        w, h = image.size
        
        # FIX: Divide 'b' by 1000.0 to convert Qwen's [0, 1000] scale to a [0.0, 1.0] scale
        x_min, y_min, x_max, y_max = [int((b / 1000.0) * w) if i % 2 == 0 else int((b / 1000.0) * h) for i, b in enumerate(bbox)]
        
        rect = plt.Rectangle((x_min, y_min), x_max-x_min, y_max-y_min, fill=False, color='red', linewidth=2)
        ax.add_patch(rect)
        ax.text(x_min, y_min-5, label, color='yellow', fontsize=10, bbox=dict(facecolor='black', alpha=0.5, pad=1))
        
    plt.axis('off')
    
    # Save the figure if a directory is provided
    if save_dir:
        base_name = os.path.basename(image_path)
        save_path = os.path.join(save_dir, f"annotated_{base_name}")
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        
    plt.show()
    plt.close() # Free up memory so the notebook doesn't crash

# Show a grid of annotated images
N = 50
for i, item in enumerate(fosae_data[:N]):
    print(f"Processing Image: {item['image']}")
    plot_image_with_boxes(os.path.join(RAW_IMAGE_DIR, item['image']), item['objects'], save_dir=ANNOTATED_DIR)
    
print(f"All annotated images saved to '{ANNOTATED_DIR}' directory.")

## 7. Export for FOSAE Training

The labeled dataset is now ready for downstream FOSAE predicate learning. The output file `fosae_labeled_dataset_unsloth.json` contains a list of images and their detected objects (class and bounding box).

In [2]:
# If on remote:
import os
from google.colab import files

# Zip the annotated images and the JSON outputs
!zip -r my_dataset_workspace.zip ./raw_images ./annotated_images unsloth_vlm_annotations.json fosae_labeled_dataset_unsloth.json dataset_metrics.png

# Trigger the browser/VS Code download prompt automatically
files.download('my_dataset_workspace.zip')